# 00 Colab 环境准备（只需运行一次）

**运行前，先在 Google 云端硬盘"我的云端硬盘"根目录放好两个文件：**
1. `ecommerce-user-behavior.zip`（项目压缩包）
2. `UserBehavior.csv.zip`（从天池下载的数据压缩包，约 900MB）

这个笔记本会：连接云端硬盘 → 解压项目 → 解压数据到 `data/raw/` → 检查文件。
完成后，在云端硬盘里打开 `ecommerce-user-behavior/notebooks/` 下的 01 到 04 依次运行。

> 云端硬盘免费空间是 15GB，整个项目大约需要 6GB（数据压缩包、解压后的 CSV、清洗后的 Parquet）。跑完 01 之后，可以删掉 `UserBehavior.csv.zip` 腾出空间。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil, zipfile
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
PROJECT = DRIVE / "ecommerce-user-behavior"
RAW_DIR = PROJECT / "data" / "raw"
RAW_CSV = RAW_DIR / "UserBehavior.csv"

# 1. 解压项目
if not PROJECT.exists():
    project_zip = DRIVE / "ecommerce-user-behavior.zip"
    assert project_zip.exists(), f"没找到 {project_zip}，请先把项目压缩包上传到云端硬盘根目录"
    zipfile.ZipFile(project_zip).extractall(DRIVE)
    print("项目已解压")
print("项目文件：", sorted(p.name for p in PROJECT.iterdir()))

In [ ]:
# 2. 解压数据（约 3.5GB，需要几分钟）
if not RAW_CSV.exists():
    data_zip = DRIVE / "UserBehavior.csv.zip"
    assert data_zip.exists(), f"没找到 {data_zip}，请先把天池下载的数据压缩包上传到云端硬盘根目录"
    tmp = Path("/content/unzipped")
    zipfile.ZipFile(data_zip).extractall(tmp)
    found = list(tmp.rglob("UserBehavior.csv"))
    assert found, "压缩包里没找到 UserBehavior.csv，请检查下载的文件"
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    shutil.move(str(found[0]), RAW_CSV)
    print("数据已解压")
print(f"UserBehavior.csv 大小：{RAW_CSV.stat().st_size / 1024**3:.2f} GB")

In [ ]:
# 3. 看一眼前 5 行，确认格式正确：用户编号, 商品编号, 类目编号, 行为类型, 时间戳
with open(RAW_CSV) as f:
    for _ in range(5):
        print(f.readline().strip())

## 下一步

在云端硬盘里依次打开：
1. `notebooks/01_数据准备与清洗.ipynb`
2. `notebooks/02_SQL_漏斗与留存分析.ipynb`
3. `notebooks/03_用户分群_RFM与聚类.ipynb`
4. `notebooks/04_购买预测模型.ipynb`

每个笔记本打开后，点菜单"代码执行程序 → 全部运行"。第一格会自动连接云端硬盘、安装需要的库和中文字体。

**建议先小规模试跑：** 第一次运行 01 时，把第三格里的 `# config.SAMPLE_FRACTION = 0.1` 去掉开头的 `#` 号，按 10% 用户抽样，确认四个笔记本都能跑通；之后再加回 `#` 号，从 01 重新跑全量。

**如果 Colab 断开连接：** 免费版闲置太久或连续运行太久会断开。云端硬盘里已经保存的文件不会丢，重新打开当前笔记本从头运行即可，不用从 01 重跑。